In [1]:
import os
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report

# OpenMP 충돌 방지
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
# ==========================================
# 1. Dual-Stage Attention Module
# ==========================================
class DualStageAttention(nn.Module):
    def __init__(self, num_features, reduction=4):
        super(DualStageAttention, self).__init__()
        hidden = max(num_features // reduction, 4)
        self.feature_mlp = nn.Sequential(
            nn.Linear(num_features * 2, hidden),
            nn.ReLU(),
            nn.Linear(hidden, num_features)
        )
        self.time_fc = nn.Sequential(
            nn.Linear(num_features, 1),
            nn.Tanh()
        )

    def forward(self, x):
        b, t, f = x.size()
        f_avg = torch.mean(x, dim=1)
        f_max, _ = torch.max(x, dim=1)
        f_cat = torch.cat([f_avg, f_max], dim=1)
        feat_score = self.feature_mlp(f_cat)
        attn_feat = torch.sigmoid(feat_score)
        x_feat_weighted = x * attn_feat.unsqueeze(1)
        time_score = self.time_fc(x_feat_weighted).squeeze(-1)
        attn_time = F.softmax(time_score, dim=1)
        x_final = x_feat_weighted * attn_time.unsqueeze(-1)
        return x_final, attn_feat, attn_time                

In [4]:
# ==========================================
# 2. Dataset Class
# ==========================================
class CANDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features) if not torch.is_tensor(features) else features
        self.labels = torch.tensor(labels) if not torch.is_tensor(labels) else labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        x = self.features[idx]
        if x.shape[0] == 9: # (9, 64) -> (64, 9) 변환
             x = x.transpose(0, 1)
        y = self.labels[idx]
        return x, y

In [5]:
# ==========================================
# 3. Explainable CAN IDS Model
# ==========================================
class ExplainableCANIDS(nn.Module):
    def __init__(self, num_features=9, seq_len=64, num_classes=5):
        super(ExplainableCANIDS, self).__init__()
        # 라벨 순서 동기화: Normal(0), DoS(1), Fuzzing(2), Replay(3), Spoofing(4)
        self.class_names = ["Normal", "DoS", "Fuzzing", "Replay", "Spoofing"]
        self.attention = DualStageAttention(num_features)
        self.cnn = nn.Sequential(
            nn.Conv1d(num_features, num_features, kernel_size=5, padding=2, groups=num_features),
            nn.BatchNorm1d(num_features),
            nn.ReLU(),
            nn.Conv1d(num_features, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )
        cnn_out_dim = 32 * (seq_len // 2) 
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(cnn_out_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x, return_attn=False):
        x_weighted, attn_feat, attn_time = self.attention(x)
        x_cnn_in = x_weighted.permute(0, 2, 1)
        features = self.cnn(x_cnn_in)
        logits = self.classifier(features)
        if return_attn:
            return logits, attn_feat, attn_time
        return logits

In [ ]:
# ==========================================
# 4. Main Execution (학습 및 평가)
#    - Early Stopping (val macro F1 기준)
#    - Test set: 모든 윈도우의 "첫 번째 패킷(t=0) feature 9개" + true/pred 저장
# ==========================================
if __name__ == "__main__":
    CONFIG = {
        "EPOCHS": 20, "BATCH_SIZE": 64, "LEARNING_RATE": 1e-3,
        "DATA_PATH": "C:/Users/user/Desktop/ids_masters/small_CAN_MIRGU/training_dataset.pt",
        "MODEL_SAVE_PATH": "best_model_final.pth"
    }
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 데이터 로드 및 6:2:2 분할
    data = torch.load(CONFIG["DATA_PATH"])
    full_dataset = CANDataset(data['X'], data['y'])
    train_size = int(0.6 * len(full_dataset))
    val_size = int(0.2 * len(full_dataset))
    test_size = len(full_dataset) - train_size - val_size
    train_ds, val_ds, test_ds = random_split(full_dataset, [train_size, val_size, test_size])

    train_loader = DataLoader(train_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=False)

    model = ExplainableCANIDS().to(device)
    optimizer = optim.Adam(model.parameters(), lr=CONFIG["LEARNING_RATE"])
    criterion = nn.CrossEntropyLoss()

    # ------------------------------
    # Early Stopping 설정 (val macro F1 기준)
    # ------------------------------
    best_val_f1 = -1.0
    best_model_wts = copy.deepcopy(model.state_dict())

    patience = 4     # 4epoch 연속 개선 없으면 stop
    min_delta = 1e-4 # 이만큼 이상 좋아져야 "개선"으로 인정
    bad_epochs = 0

    print(f"[*] 학습 시작... (Device: {device})")
    for epoch in range(CONFIG["EPOCHS"]):
        # -------- train --------
        model.train()
        t_loss = 0.0
        for x_b, y_b in train_loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x_b), y_b)
            loss.backward()
            optimizer.step()
            t_loss += loss.item()

        # -------- val --------
        model.eval()
        v_loss, v_preds, v_targets = 0.0, [], []
        with torch.no_grad():
            for x_v, y_v in val_loader:
                x_v, y_v = x_v.to(device), y_v.to(device)
                logits = model(x_v)
                v_loss += criterion(logits, y_v).item()
                v_preds.extend(torch.argmax(logits, 1).cpu().numpy())
                v_targets.extend(y_v.cpu().numpy())

        # 라벨별 F1 계산 및 출력
        f1_macro = f1_score(v_targets, v_preds, average='macro')
        f1_per_class = f1_score(v_targets, v_preds, average=None, labels=[0, 1, 2, 3, 4])

        print(f"Epoch {epoch+1:02d} | Train Loss: {t_loss/len(train_loader):.4f} | "
              f"Val Loss: {v_loss/len(val_loader):.4f} | Macro F1: {f1_macro:.4f}")
        detail = " > Details: " + " ".join([f"[{model.class_names[i]}: {f1_per_class[i]:.3f}]" for i in range(5)])
        print(detail + "\n" + "-"*100)

        # ------------------------------
        # Early Stopping 체크 (val macro F1)
        # ------------------------------
        if f1_macro > best_val_f1 + min_delta:
            best_val_f1 = f1_macro
            best_model_wts = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            print(f"[!] No val improvement: {bad_epochs}/{patience}")

        if bad_epochs >= patience:
            print(f"[EarlyStopping] Stop at epoch {epoch+1} (best macro F1={best_val_f1:.4f})")
            break

    # ------------------------------
    # 윈도우 첫번째 패킷(t=0) feature 뽑기 함수
    # ------------------------------
    def get_first_packet_features(x_win: torch.Tensor) -> np.ndarray:
        x_win = x_win.detach().cpu()
        if x_win.ndim != 2:
            raise ValueError(f"Expected 2D tensor per window, got shape={tuple(x_win.shape)}")

        # (9, 64) => features x time
        if x_win.shape[0] == 9:
            first_pkt = x_win[:, 0]
        # (64, 9) => time x features
        elif x_win.shape[1] == 9:
            first_pkt = x_win[0, :]
        else:
            raise ValueError(f"Cannot infer feature dimension=9 from shape={tuple(x_win.shape)}")

        return first_pkt.numpy()

    # ------------------------------
    # 최종 평가 (Test Set) + 첫 패킷 feature 저장
    # ------------------------------
    print("\n[*] 최종 테스트 결과 (Test Set)")
    model.load_state_dict(best_model_wts)
    model.eval()

    t_preds, t_targets = [], []

    f_names = ["G_IAT", "ID_IAT", "Ent", "Ham", "DLC", "Delta", "Jit", "Mean", "Std"]
    window_records = []
    global_window_idx = 0

    with torch.no_grad():
        for x_t, y_t in test_loader:
            x_t, y_t = x_t.to(device), y_t.to(device)

            logits = model(x_t)
            preds = torch.argmax(logits, dim=1)

            t_preds.extend(preds.cpu().numpy())
            t_targets.extend(y_t.cpu().numpy())

            # 배치 내 모든 윈도우 기록
            for i in range(x_t.size(0)):
                first_pkt_feat = get_first_packet_features(x_t[i])
                true_i = int(y_t[i].item())
                pred_i = int(preds[i].item())

                rec = {
                    "window_idx": global_window_idx,
                    "true_label": true_i,
                    "true_name": model.class_names[true_i],
                    "pred_label": pred_i,
                    "pred_name": model.class_names[pred_i],
                }
                for k, fn in enumerate(f_names):
                    rec[fn] = float(first_pkt_feat[k])

                window_records.append(rec)
                global_window_idx += 1

    print(classification_report(t_targets, t_preds, target_names=model.class_names, digits=4))

    # 저장 (전체 + pred별)
    df_firstpkt = pd.DataFrame(window_records)

    out_csv = "test_windows_first_packet_features_ALL.csv"
    df_firstpkt.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"\n[+] Saved ALL windows first-packet features -> {out_csv}")

    for c in range(len(model.class_names)):
        name = model.class_names[c]
        df_c = df_firstpkt[df_firstpkt["pred_label"] == c].copy()
        out_c = f"test_windows_first_packet_features_PRED_{name}.csv"
        df_c.to_csv(out_c, index=False, encoding="utf-8-sig")
        print(f"[+] Saved PRED={name} windows -> {out_c} (n={len(df_c)})")

    # ------------------------------
    # XAI 분석 (샘플별) - 기존 로직 유지
    # ------------------------------
    print("="*30 + " XAI Analysis " + "="*30)
    for lbl_idx in [1, 2, 3, 4]:
        for x, y in test_ds:
            if int(y) == lbl_idx:
                x_in = x.unsqueeze(0).to(device)
                logits, feat_imp, time_imp = model(x_in, return_attn=True)
                top_f = np.argsort(feat_imp[0].cpu().detach().numpy())[::-1][:3]
                f_names2 = ["G_IAT", "ID_IAT", "Ent", "Ham", "DLC", "Delta", "Jit", "Mean", "Std"]
                print(f"[{model.class_names[lbl_idx]}] -> Top Features: {[f_names2[i] for i in top_f]}")
                break


[*] 학습 시작... (Device: cuda)
Epoch 01 | Train Loss: 0.0124 | Val Loss: 1.7983 | Macro F1: 0.2528
 > Details: [Normal: 0.678] [DoS: 0.000] [Fuzzing: 0.000] [Replay: 0.530] [Spoofing: 0.056]
----------------------------------------------------------------------------------------------------
Epoch 02 | Train Loss: 0.0017 | Val Loss: 1.1964 | Macro F1: 0.1989
 > Details: [Normal: 0.979] [DoS: 0.000] [Fuzzing: 0.015] [Replay: 0.000] [Spoofing: 0.000]
----------------------------------------------------------------------------------------------------
[!] No val improvement: 1/4
Epoch 03 | Train Loss: 0.0012 | Val Loss: 1.1127 | Macro F1: 0.3262
 > Details: [Normal: 0.983] [DoS: 0.083] [Fuzzing: 0.027] [Replay: 0.015] [Spoofing: 0.523]
----------------------------------------------------------------------------------------------------
Epoch 04 | Train Loss: 0.0009 | Val Loss: 0.1994 | Macro F1: 0.6508
 > Details: [Normal: 0.989] [DoS: 0.190] [Fuzzing: 0.149] [Replay: 0.985] [Spoofing: 0.941]
-

RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.